# Kandinsky Game→Audio — Step 7: Post-Load Adapter Across a Tiny DiT

Step 6 showed that a Game→Audio adapter can be attached to a real fused block after strict-loading the base weights.

This step extends the same pattern to a **tiny full DiffusionTransformer3D**:
1. Create a synthetic base state dict.
2. Build the expanded model on the meta device with no game branch, then strict-load the base state.
3. Materialize only the model's non-persistent buffers.
4. Attach new Game→Audio modules to each fused block.
5. Save and restore only the adapter sidecar.
6. Compare audio-only and joint video+audio forward paths.

**No pretrained checkpoint, training, real audio, or pipeline generation is used.** A CPU runtime is sufficient. Start with a fresh or restarted Colab runtime so the isolated Kandinsky imports from earlier notebooks do not conflict.

## Expected result

With all new gates closed, the expanded model should match the base model. Opening only the last block's gate should change audio. The sidecar-restored model should reproduce the same output, while the direct video output remains unchanged for this last-block-only change.

## 1. Environment

In [ ]:
import importlib.util
import subprocess
import sys
import types
from pathlib import Path

import torch
import torch.nn.functional as F

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device: CPU is sufficient for this small forward-path check.")

## 2. Pin the source revision

In [ ]:
K6_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
K6_DIR = Path("/content/kandinsky-6-step7-lab")

if not K6_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/kandinskylab/kandinsky-6.git", str(K6_DIR)
    ], check=True)

current = subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip()
if current != K6_COMMIT:
    subprocess.run(
        ["git", "-C", str(K6_DIR), "fetch", "--depth", "1", "origin", K6_COMMIT],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(K6_DIR), "checkout", "--detach", K6_COMMIT],
        check=True,
    )

DIT_PATH = K6_DIR / "kandinsky/core/components/dit.py"
source = DIT_PATH.read_text(encoding="utf-8")
assert subprocess.check_output(
    ["git", "-C", str(K6_DIR), "rev-parse", "HEAD"], text=True
).strip() == K6_COMMIT
print("Pinned source:", K6_COMMIT)
print("Original source:", DIT_PATH)

## 3. Patch only a scratch source copy

The patch is the same full-DiT Game→Audio plumbing used in Step 4. It supports game tokens but does not instantiate the branch unless a game dimension is configured. Here we deliberately configure no branch until **after** strict base loading.

In [ ]:
def replace_once(text, old, new, label):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f"Expected one source anchor for {label}; found {count}.")
    return text.replace(old, new, 1)

patched = source

# Add the optional event-token width to the real fused block.
patched = replace_once(
    patched,
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n    ):\n",
    "        ca_rope: bool = False, cross_gates: bool = False, fix_modulation: bool = False,\n"
    "        game_dim: int | None = None,\n    ):\n",
    "fused-block constructor",
)

# Create the Game→Audio attention branch only when game_dim is configured.
patched = replace_once(
    patched,
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n",
    "        self.model_dim     = model_dim\n        self.model_dim_a   = model_dim_a\n"
    "        self.game_dim      = game_dim\n"
    "        if game_dim is None:\n"
    "            self.game_audio_norm = None\n"
    "            self.game_to_audio = None\n"
    "            self.game_audio_modulation = None\n"
    "        else:\n"
    "            self.game_audio_norm = nn.LayerNorm(model_dim_a, elementwise_affine=False)\n"
    "            self.game_to_audio = MultiheadCrossAttention(model_dim_a, head_dim_a, game_dim, engine)\n"
    "            self.game_audio_modulation = Modulation(time_dim_a, model_dim_a, 1)\n",
    "fused-block Game→Audio modules",
)

# Extend the fused block call. These are also used for audio-only calls.
patched = replace_once(
    patched,
    "        av_gate_scale: float = 1.0,\n"
    "        va_gate_scale: float = 1.0,\n"
    "    ):\n",
    "        av_gate_scale: float = 1.0,\n"
    "        va_gate_scale: float = 1.0,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ):\n",
    "fused-block forward signature",
)

# Apply the branch after existing text/AV updates and before the audio FFN.
patched = replace_once(
    patched,
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # ---- FFN ----\n",
    "        else:\n"
    "            vis = apply_gate_sum(vis, vis_out_t, gate_v).type_as(vis)\n"
    "\n"
    "        # Game events condition the audio stream in both audio-only and AV calls.\n"
    "        if game_tokens is not None:\n"
    "            if aud is None:\n"
    "                raise ValueError(\"Game conditioning requires an audio stream.\")\n"
    "            if self.game_to_audio is None:\n"
    "                raise ValueError(\"Construct this block with game_dim to accept game_tokens.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, tokens, game_dim].\")\n"
    "            if game_tokens.shape[0] != aud.shape[0]:\n"
    "                raise ValueError(\"game_tokens and audio must have the same batch size.\")\n"
    "            game_tokens = game_tokens.to(\n"
    "                device=aud.device, dtype=self.game_to_audio.to_key.weight.dtype\n"
    "            )\n"
    "            if game_attn_mask is not None:\n"
    "                game_attn_mask = game_attn_mask.to(device=aud.device, dtype=torch.bool)\n"
    "            game_query = self.game_audio_norm(aud).type_as(aud)\n"
    "            game_update = self.game_to_audio(game_query, game_tokens, game_attn_mask)\n"
    "            game_gate = self.game_audio_modulation(t_a)\n"
    "            aud = apply_gate_sum(aud, game_update, game_gate).type_as(aud)\n"
    "\n"
    "        # ---- FFN ----\n",
    "fused-block Game→Audio insertion",
)

# Add game_dim to DiffusionTransformer3D and pass it to every fused block.
patched = replace_once(
    patched,
    "        visual_token_type_num_embeddings: int = 0,\n"
    "    ):\n",
    "        visual_token_type_num_embeddings: int = 0,\n"
    "        game_dim: int | None = None,\n"
    "    ):\n",
    "DiT constructor signature",
)
patched = replace_once(
    patched,
    "        self.is_multimodal  = is_multimodal\n"
    "        self.in_visual_dim  = in_visual_dim\n",
    "        self.is_multimodal  = is_multimodal\n"
    "        self.game_dim       = game_dim\n"
    "        if game_dim is not None and not is_multimodal:\n"
    "            raise ValueError(\"game_dim is supported only by the multimodal audio model.\")\n"
    "        self.in_visual_dim  = in_visual_dim\n",
    "DiT game_dim configuration",
)
patched = replace_once(
    patched,
    "                    ca_rope=ca_rope, cross_gates=cross_gates, fix_modulation=fix_modulation,\n",
    "                    ca_rope=ca_rope, cross_gates=cross_gates, fix_modulation=fix_modulation,\n"
    "                    game_dim=game_dim,\n",
    "pass game_dim to fused-block stack",
)

# Thread game tokens through the audio-only fused-block loop.
patched = replace_once(
    patched,
    "        aud_rope: Tensor | None,\n"
    "        attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor | None, Tensor | None]:\n",
    "        aud_rope: Tensor | None,\n"
    "        attn_mask: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor | None, Tensor | None]:\n",
    "single-modality block-loop signature",
)
patched = replace_once(
    patched,
    "                    _, aud_embed = blk(\n"
    "                        None, aud_embed, te, te, (tm, tm), None, aud_rope, attn_mask,\n"
    "                    )\n",
    "                    _, aud_embed = blk(\n"
    "                        None, aud_embed, te, te, (tm, tm), None, aud_rope, attn_mask,\n"
    "                        game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "                    )\n",
    "audio-only fused-block call",
)

# Thread the same tensors through the joint video+audio loop.
patched = replace_once(
    patched,
    "        aud_rope: Tensor,\n"
    "        attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor, Tensor]:\n",
    "        aud_rope: Tensor,\n"
    "        attn_mask: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> tuple[Tensor, Tensor]:\n",
    "fused block-loop signature",
)
patched = replace_once(
    patched,
    "                vis_rope, aud_rope,\n"
    "                attn_mask,\n"
    "            )\n"
    "        return vis_embed, aud_embed\n",
    "                vis_rope, aud_rope,\n"
    "                attn_mask,\n"
    "                game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "            )\n"
    "        return vis_embed, aud_embed\n",
    "joint fused-block call",
)

# Add public forward arguments and validate their basic contract.
patched = replace_once(
    patched,
    "        attention_mask: Tensor | None = None,\n"
    "        visual_token_type_ids: Tensor | None = None,\n"
    "    ) -> Tensor | tuple[Tensor, Tensor]:\n",
    "        attention_mask: Tensor | None = None,\n"
    "        visual_token_type_ids: Tensor | None = None,\n"
    "        game_tokens: Tensor | None = None,\n"
    "        game_attn_mask: Tensor | None = None,\n"
    "    ) -> Tensor | tuple[Tensor, Tensor]:\n",
    "public forward signature",
)
patched = replace_once(
    patched,
    "        both = x_video is not None and x_audio is not None and self.is_multimodal\n"
    "        attn_mask = self._normalize_attn_mask(attention_mask)\n",
    "        both = x_video is not None and x_audio is not None and self.is_multimodal\n"
    "        if game_attn_mask is not None and game_tokens is None:\n"
    "            raise ValueError(\"game_attn_mask requires game_tokens.\")\n"
    "        if game_tokens is not None:\n"
    "            if not self.is_multimodal or self.game_dim is None:\n"
    "                raise ValueError(\"Configure game_dim on a multimodal model before passing game_tokens.\")\n"
    "            if x_audio is None:\n"
    "                raise ValueError(\"Game conditioning requires x_audio.\")\n"
    "            if game_tokens.ndim != 3 or game_tokens.shape[-1] != self.game_dim:\n"
    "                raise ValueError(\"game_tokens must have shape [batch, events, game_dim].\")\n"
    "            audio_batch = x_audio.shape[0] if x_audio.ndim == 3 else 1\n"
    "            if game_tokens.shape[0] != audio_batch:\n"
    "                raise ValueError(\"game_tokens and x_audio must have the same batch size.\")\n"
    "            if game_attn_mask is not None:\n"
    "                if game_attn_mask.dtype != torch.bool:\n"
    "                    raise ValueError(\"game_attn_mask must be boolean; True means a valid event token.\")\n"
    "                if game_attn_mask.shape[-1] != game_tokens.shape[1]:\n"
    "                    raise ValueError(\"The final game_attn_mask dimension must match the event count.\")\n"
    "        attn_mask = self._normalize_attn_mask(attention_mask)\n",
    "forward argument validation",
)

# Pass the optional condition to audio-only and joint forward paths.
patched = replace_once(
    patched,
    "                None, aud_embed, te, tm, None, aud_rope, attn_mask,\n"
    "            )\n"
    "            return self._project_audio(aud_embed, tm)\n",
    "                None, aud_embed, te, tm, None, aud_rope, attn_mask,\n"
    "                game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "            )\n"
    "            return self._project_audio(aud_embed, tm)\n",
    "audio-only DiT call",
)
patched = replace_once(
    patched,
    "            vis_rope, aud_rope, attn_mask,\n"
    "        )\n"
    "        return self._project_fused(\n",
    "            vis_rope, aud_rope, attn_mask,\n"
    "            game_tokens=game_tokens, game_attn_mask=game_attn_mask,\n"
    "        )\n"
    "        return self._project_fused(\n",
    "joint DiT call",
)

PATCH_PATH = DIT_PATH.with_name("dit_game_full.py")
PATCH_PATH.write_text(patched, encoding="utf-8")
print("Patched scratch copy:", PATCH_PATH)
print("Original source unchanged:", DIT_PATH.read_text(encoding="utf-8") == source)

## 4. Import the pinned model code

In [ ]:
if "kandinsky" in sys.modules:
    raise RuntimeError(
        "A Kandinsky package is already imported. Restart the runtime, then run all cells from the top."
    )

def register_package(name, path):
    module = types.ModuleType(name)
    module.__path__ = [str(path)]
    module.__package__ = name
    sys.modules[name] = module

register_package("kandinsky", K6_DIR / "kandinsky")
register_package("kandinsky.core", K6_DIR / "kandinsky/core")
register_package("kandinsky.core.components", K6_DIR / "kandinsky/core/components")
register_package("kandinsky.core.components.attention", K6_DIR / "kandinsky/core/components/attention")
register_package("kandinsky.runtime", K6_DIR / "kandinsky/runtime")
register_package("kandinsky.runtime.kernels", K6_DIR / "kandinsky/runtime/kernels")

def pinned_sdpa(q, k, v, attn_mask=None):
    # Kandinsky attention tensors are [batch, sequence, heads, head_dim].
    q = q.transpose(1, 2).contiguous()
    k = k.transpose(1, 2).contiguous()
    v = v.transpose(1, 2).contiguous()
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask).transpose(1, 2).contiguous()

class SDPAOnlyEngine:
    def __init__(self, engine="auto"):
        if engine not in {"auto", "sdpa"}:
            raise ValueError("This notebook supports only the SDPA attention path.")
    def get_attention(self):
        return pinned_sdpa

engine_module = types.ModuleType("kandinsky.runtime.kernels.attention_engine")
engine_module._sdpa = pinned_sdpa
engine_module.SelfAttentionEngine = SDPAOnlyEngine
engine_module.resolve_attention_engine = lambda requested: "sdpa"
sys.modules[engine_module.__name__] = engine_module

def load_dit_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

base_dit = load_dit_module(
    "kandinsky.core.components.dit_base", DIT_PATH
)
game_dit = load_dit_module(
    "kandinsky.core.components.dit_game_full", PATCH_PATH
)
print("Baseline model class:", base_dit.DiffusionTransformer3D)
print("Patched model class:", game_dit.DiffusionTransformer3D)

## 5. Build a tiny base model and synthetic checkpoint

The checkpoint is generated locally from a small random model; it is not downloaded. It represents the original Kandinsky state dict before any game modules exist.

In [ ]:
torch.manual_seed(707)

BASE_CONFIG = dict(
    in_visual_dim=4,
    out_visual_dim=4,
    in_text_dim=12,
    in_text_dim2=6,
    time_dim=16,
    patch_size=(1, 2, 2),
    model_dim=24,
    ff_dim=48,
    num_text_blocks=1,
    num_visual_blocks=2,
    axes_dims=(2, 2, 2),
    visual_cond=False,
    is_multimodal=True,
    in_audio_dim=5,
    out_audio_dim=5,
    model_dim_a=24,
    time_dim_a=16,
    ff_dim_a=48,
    axes_dims_a=(2, 2, 2),
    attention_engine="sdpa",
)
GAME_DIM = 7
DEVICE = torch.device("cpu")
DTYPE = torch.float32

base_model = base_dit.DiffusionTransformer3D(**BASE_CONFIG).eval()
base_checkpoint = {
    key: value.detach().cpu().clone()
    for key, value in base_model.state_dict().items()
}
print("Synthetic base state keys:", len(base_checkpoint))
print("Fused blocks:", len(base_model.visual_transformer_blocks))

## 6. Strict-load the base first on the meta device

The DiT has non-persistent RoPE and timestep buffers that are not stored in checkpoints. Kandinsky's factory materializes those buffers after loading. We mirror that source helper here; it does not allocate missing parameters.

In [ ]:
with torch.device("meta"):
    expanded = game_dit.DiffusionTransformer3D(
        **BASE_CONFIG,
        game_dim=None,
    )

base_load_result = expanded.load_state_dict(
    base_checkpoint,
    strict=True,
    assign=True,
)
assert not base_load_result.missing_keys
assert not base_load_result.unexpected_keys
assert all(not parameter.is_meta for parameter in expanded.parameters())

meta_buffers_before = [
    f"{module_name}.{buffer_name}"
    for module_name, module in expanded.named_modules()
    for buffer_name, buffer in module._buffers.items()
    if buffer is not None and buffer.is_meta
]
print("Meta buffers before reconstruction:", len(meta_buffers_before))
assert meta_buffers_before, "Expected non-persistent RoPE/time buffers on meta."

def materialize_meta_buffers(module, device):
    for submodule in module.modules():
        touched = False
        for name, buffer in list(submodule._buffers.items()):
            if buffer is not None and buffer.device.type == "meta":
                submodule._buffers[name] = torch.empty(
                    buffer.shape,
                    dtype=buffer.dtype,
                    device=device,
                )
                touched = True
        if touched and hasattr(submodule, "reset_parameters"):
            submodule.reset_parameters()

materialize_meta_buffers(expanded, DEVICE)
assert all(not buffer.is_meta for buffer in expanded.buffers() if buffer is not None)
expanded = expanded.to(DEVICE).eval()
print("Strict base state loaded: passed")
print("Non-persistent meta buffers materialized: passed")

## 7. Attach the Game→Audio modules after loading

We create new layers directly on CPU in the base model's dtype. The base checkpoint loader never sees these new parameters, so its strict key check remains exact.

In [ ]:
def attach_game_adapter(model, game_dim):
    if not model.is_multimodal:
        raise ValueError("Game→Audio attachment requires a multimodal DiT.")
    if model.game_dim is not None:
        raise RuntimeError("A game adapter is already configured.")

    model.game_dim = game_dim
    for block in model.visual_transformer_blocks:
        if block.game_to_audio is not None:
            raise RuntimeError("A fused block already has a Game→Audio branch.")

        block.game_dim = game_dim
        block.game_audio_norm = torch.nn.LayerNorm(
            block.model_dim_a,
            elementwise_affine=False,
            device=DEVICE,
            dtype=DTYPE,
        )
        block.game_to_audio = game_dit.MultiheadCrossAttention(
            block.model_dim_a,
            sum(BASE_CONFIG["axes_dims_a"]),
            game_dim,
            "sdpa",
        ).to(device=DEVICE, dtype=DTYPE)
        block.game_audio_modulation = game_dit.Modulation(
            BASE_CONFIG["time_dim_a"],
            block.model_dim_a,
            1,
        ).to(device=DEVICE, dtype=DTYPE)
        block.eval()
    return model

attach_game_adapter(expanded, GAME_DIM)
assert expanded.game_dim == GAME_DIM
assert len(expanded.visual_transformer_blocks) == BASE_CONFIG["num_visual_blocks"]
assert all(block.game_to_audio is not None for block in expanded.visual_transformer_blocks)
print("Attached branches:", len(expanded.visual_transformer_blocks))
print("No parameters left on meta:", all(not p.is_meta for p in expanded.parameters()))

## 8. Verify original weights and save a sidecar

We open only the final block's gate manually for a serialization demo, then save the new branch modules only. The gate value is not learned.

In [ ]:
def assert_base_weights_unchanged(model):
    current = model.state_dict()
    for key, expected in base_checkpoint.items():
        torch.testing.assert_close(
            current[key].detach().cpu(),
            expected,
            rtol=0,
            atol=0,
        )

assert_base_weights_unchanged(expanded)
print("Base model parameters preserved: passed")

with torch.no_grad():
    expanded.visual_transformer_blocks[-1].game_audio_modulation.out_layer.bias.fill_(0.1)

def collect_adapter_state(model):
    saved = {"game_dim": int(model.game_dim), "blocks": {}}
    for index, block in enumerate(model.visual_transformer_blocks):
        saved["blocks"][str(index)] = {
            module_name: {
                key: value.detach().cpu().clone()
                for key, value in getattr(block, module_name).state_dict().items()
            }
            for module_name in (
                "game_audio_norm",
                "game_to_audio",
                "game_audio_modulation",
            )
        }
    return saved

adapter_state = collect_adapter_state(expanded)
adapter_bytes = io.BytesIO()
torch.save(adapter_state, adapter_bytes)
adapter_bytes.seek(0)
adapter_restored = torch.load(
    adapter_bytes,
    map_location="cpu",
    weights_only=True,
)
assert adapter_restored["game_dim"] == GAME_DIM
assert len(adapter_restored["blocks"]) == BASE_CONFIG["num_visual_blocks"]
print("Sidecar blocks:", len(adapter_restored["blocks"]))
print("Adapter sidecar created without base-model weights: passed")

## 9. Strict-load a second base, attach, and restore the sidecar

In [ ]:
with torch.device("meta"):
    restored = game_dit.DiffusionTransformer3D(
        **BASE_CONFIG,
        game_dim=None,
    )

restore_result = restored.load_state_dict(
    base_checkpoint,
    strict=True,
    assign=True,
)
assert not restore_result.missing_keys
assert not restore_result.unexpected_keys
materialize_meta_buffers(restored, DEVICE)
restored = restored.to(DEVICE).eval()
attach_game_adapter(restored, GAME_DIM)

if adapter_restored["game_dim"] != restored.game_dim:
    raise ValueError("Adapter game_dim does not match this model.")
if len(adapter_restored["blocks"]) != len(restored.visual_transformer_blocks):
    raise ValueError("Adapter block count does not match this model.")

for index, block in enumerate(restored.visual_transformer_blocks):
    saved = adapter_restored["blocks"][str(index)]
    for module_name, state in saved.items():
        getattr(block, module_name).load_state_dict(state, strict=True)

assert_base_weights_unchanged(restored)
print("Strict base load + strict per-module sidecar restore: passed")
print("Base weights unchanged after sidecar restore: passed")

## 10. Compare audio-only and joint video+audio paths

The base and adapter models receive identical miniature latents and conditions. All gates start closed, so both routes should preserve the base output.

In [ ]:
BATCH = 2
AUDIO_LEN = 8
TEXT_LEN = 4
GAME_LEN = 5
VIDEO_FRAMES = 2

x_audio = torch.randn(BATCH, AUDIO_LEN, BASE_CONFIG["in_audio_dim"])
audio_text = torch.randn(BATCH, TEXT_LEN, BASE_CONFIG["in_text_dim"])
audio_pooled = torch.randn(BATCH, BASE_CONFIG["in_text_dim2"])
audio_time = torch.tensor([0.15, 0.8])

x_video = torch.randn(BATCH, VIDEO_FRAMES, 2, 2, BASE_CONFIG["in_visual_dim"])
video_text = torch.randn(BATCH, TEXT_LEN, BASE_CONFIG["in_text_dim"])
video_pooled = torch.randn(BATCH, BASE_CONFIG["in_text_dim2"])
video_time = torch.tensor([0.25, 0.65])

game_tokens = torch.randn(BATCH, GAME_LEN, GAME_DIM)
game_valid = torch.tensor([
    [True, True, True, False, False],
    [True, True, False, False, False],
])
game_attn_mask = game_valid[:, None, None, :]

audio_rope = expanded.audio_rope(torch.arange(AUDIO_LEN))
text_positions = torch.arange(TEXT_LEN)
audio_text_rope = expanded.audio_text_rope(text_positions)
video_grid = (VIDEO_FRAMES, 1, 1)
video_positions = [torch.arange(VIDEO_FRAMES), torch.arange(1), torch.arange(1)]
video_rope = expanded.visual_rope(video_grid, video_positions)
video_text_rope = expanded.video_text_rope(text_positions)

joint_text = [video_text, audio_text]
joint_pooled = [video_pooled, audio_pooled]
joint_time = [video_time, audio_time]
joint_text_rope = [video_text_rope, audio_text_rope]

# Before changing any gates, verify that the sidecar's open gate
# reproduces the source model in both routes.
with torch.no_grad():
    source_sidecar_audio = expanded(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    restored_sidecar_audio = restored(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    source_sidecar_joint = expanded(
        x_video, x_audio, joint_text, joint_pooled, joint_time,
        video_rope, audio_rope, joint_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    restored_sidecar_joint = restored(
        x_video, x_audio, joint_text, joint_pooled, joint_time,
        video_rope, audio_rope, joint_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(restored_sidecar_audio, source_sidecar_audio, rtol=0, atol=0)
torch.testing.assert_close(restored_sidecar_joint[0], source_sidecar_joint[0], rtol=0, atol=0)
torch.testing.assert_close(restored_sidecar_joint[1], source_sidecar_joint[1], rtol=0, atol=0)
print("Sidecar-restored open-gate outputs match: passed")

# Close all gates to establish exact base-model parity.
with torch.no_grad():
    for model in (expanded, restored):
        for block in model.visual_transformer_blocks:
            block.game_audio_modulation.out_layer.bias.zero_()

with torch.no_grad():
    base_audio = base_model(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
    )
    expanded_audio_closed = expanded(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    restored_audio_closed = restored(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(expanded_audio_closed, base_audio, rtol=0, atol=0)
torch.testing.assert_close(restored_audio_closed, base_audio, rtol=0, atol=0)
print("Audio-only zero-gate parity: passed")

with torch.no_grad():
    base_joint = base_model(
        x_video, x_audio, joint_text, joint_pooled, joint_time,
        video_rope, audio_rope, joint_text_rope,
    )
    expanded_joint_closed = expanded(
        x_video, x_audio, joint_text, joint_pooled, joint_time,
        video_rope, audio_rope, joint_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    restored_joint_closed = restored(
        x_video, x_audio, joint_text, joint_pooled, joint_time,
        video_rope, audio_rope, joint_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

torch.testing.assert_close(expanded_joint_closed[0], base_joint[0], rtol=0, atol=0)
torch.testing.assert_close(expanded_joint_closed[1], base_joint[1], rtol=0, atol=0)
torch.testing.assert_close(restored_joint_closed[0], base_joint[0], rtol=0, atol=0)
torch.testing.assert_close(restored_joint_closed[1], base_joint[1], rtol=0, atol=0)
print("Joint video+audio zero-gate parity: passed")

## 11. Open only the final block and verify sidecar equivalence

Earlier blocks remain closed. In the final block, the Game→Audio branch is applied after that block's existing video/audio exchange, so the direct video output should remain unchanged in this isolated check.

In [ ]:
with torch.no_grad():
    expanded.visual_transformer_blocks[-1].game_audio_modulation.out_layer.bias.fill_(0.1)
    restored.visual_transformer_blocks[-1].game_audio_modulation.out_layer.bias.fill_(0.1)

with torch.no_grad():
    expanded_audio_open = expanded(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    expanded_joint_open = expanded(
        x_video, x_audio, joint_text, joint_pooled, joint_time,
        video_rope, audio_rope, joint_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    restored_audio_open = restored(
        None, x_audio, audio_text, audio_pooled, audio_time,
        None, audio_rope, audio_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )
    restored_joint_open = restored(
        x_video, x_audio, joint_text, joint_pooled, joint_time,
        video_rope, audio_rope, joint_text_rope,
        game_tokens=game_tokens,
        game_attn_mask=game_attn_mask,
    )

audio_delta = (expanded_audio_open - base_audio).abs().mean().item()
joint_audio_delta = (expanded_joint_open[1] - base_joint[1]).abs().mean().item()
torch.testing.assert_close(expanded_joint_open[0], base_joint[0], rtol=0, atol=0)
torch.testing.assert_close(restored_audio_open, expanded_audio_open, rtol=0, atol=0)
torch.testing.assert_close(restored_joint_open[0], expanded_joint_open[0], rtol=0, atol=0)
torch.testing.assert_close(restored_joint_open[1], expanded_joint_open[1], rtol=0, atol=0)
assert audio_delta > 0 and joint_audio_delta > 0

print("Audio-only mean change:", audio_delta)
print("Joint audio mean change:", joint_audio_delta)
print("Last-block video output unchanged: passed")
print("Restored sidecar reproduces audio-only and joint outputs: passed")

## What Step 7 proves — and does not prove

**Proves if all cells pass:** a tiny full DiT can strict-load its original synthetic base state on meta, materialize its non-persistent buffers, receive new Game→Audio modules after loading, and restore only those modules from a sidecar. Both audio-only and joint video+audio paths keep base output when the gates are closed.

**Does not prove:** compatibility with a real 26+ GB checkpoint, the official pipeline/offload/compile wrappers, training, event-token quality, or audible control. The model is tiny and random; the open gate is manual.

**Next:** once this adapter lifecycle is verified, integrate the condition argument into one real sampler path behind a small local patch, still without training or production weights.